In [1]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import re
# from io import BytesIO
import requests
import sparse
import time
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

# import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

# coiled notebook start --region=us-east-1

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
!aws configure set aws_access_key_id [access key]
!aws configure set aws_secret_access_key [secret key]

In [4]:
cluster = coiled.Cluster(
    name="tcl_dask",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=50,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    scheduler_vm_types="x2gd.xlarge", 
    worker_vm_types="x2gd.xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.2.2.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.10 linux-aarch64 on conda-forge   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.10 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [5]:
# Conversion of carbon to CO2
C_to_CO2 = 44/12

def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [6]:
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Try fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [7]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [8]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 
    2222100, 2222200, 2223100, 2223200, 3110000, 3120000, 
    3211111, 3211112, 3211121, 3211122, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [9]:
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

In [10]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [11]:
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()

    return xarray_chunks

In [12]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(nodes, analysis_layer, join="inner")
    return analysis_layer_sub, nodes_aligned

In [13]:
def xarray_reduction_sum_count(analysis_layer, node_data):

    reductions = {}

    for func in ["sum", "count"]:
        reduced = xarray_reduce(
            analysis_layer.band_data,
            node_data,
            func=func,
            keep_attrs=True,
            expected_groups=(node_codes),
            reindex=ReindexStrategy(
                blockwise=False,
                array_type=ReindexArrayType.SPARSE_COO
            ),
            fill_value=0
        )

        # Rename variables to reflect reduction type
        if isinstance(reduced, xr.Dataset):
            renamed = reduced.rename({var: f"{var}_{func}" for var in reduced.data_vars})
        else:  # it's a DataArray
            renamed = reduced.rename(f"{reduced.name}_{func}")

        reductions[func] = renamed

    # Merge results: handle Dataset or DataArray combinations
    result = xr.merge([r if isinstance(r, xr.Dataset) else r.to_dataset() for r in reductions.values()])

    return result

In [14]:
def xarray_reduction(flux_cube, nodes_aligned_data, adm0_data):

    data_cube_by_node = xarray_reduce(
        flux_cube,
        nodes_aligned_data,
        adm0_data,
        func='sum',
        keep_attrs=True,
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return data_cube_by_node

In [15]:
# Converts flox output to dataframe and does some processing of it
def create_interval_df(coord_dict, state_node_df):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Replaces numeric values for outputs with names
    df['flux_type'] = df['flux_type'].replace({0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 
                                               2: gross_remv_all_pools_output_pattern, 3: net_flux_output_pattern, 4: "area__ha"})
    # print("with flux_type:", df)
    
    # Classifies the node_codes by larger groupings
    df['node_grp'] = df['state_node'].apply(classify_node)
    # print("with classified nodes:", df)

    # Makes node_codes into strings
    df['state_node'] = 'n' + df['state_node'].astype(str)
    # print("with n prefix:", df)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    # print("with interval end year:", df)

    df = df.merge(state_node_df[['state_node_with_prefix', 'meaning']],
              left_on='state_node', right_on='state_node_with_prefix',
              how='left')

    # Converts area from m^2 to ha
    df.loc[df['flux_type'].eq('area__ha'), 'value'] = df['value'] / 10000

    # Drop the helper column if you don't want it
    df.drop(columns=['state_node_with_prefix'], inplace=True)
    
    # print(df)

    return df

In [16]:
# Calculates flux densities (Mg CO2 or CO2e/ha)
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['flux_type'] == 'area__ha'].copy()
    flux_df = df[df['flux_type'] != 'area__ha'].copy()
    
    # Step 2: Merges flux data with area data on matching keys
    merged = pd.merge(
        flux_df,
        area_df[['state_node', 'gadm_adm0', 'interval_end', 'value']],
        on=['state_node', 'gadm_adm0', 'interval_end'],
        how='left',
        suffixes=('', '_area')
    )
    # print("merged:" merged)
    
    # Step 3: Computes per-hectare flux (converts CO2 to C)
    merged['value_per_ha'] = merged['value'] / merged['value_area'] / C_to_CO2
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['flux_type'] = new_rows['flux_type'] + '__C_per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
    # print("new rows:", new_rows)
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

    return result_df

In [17]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [18]:
# uri components

# model_version = "version_0_3_2"
# run_date = "20250507"
# chunk_size = 4000

model_version = "version_0_3_3"
run_date = "20250511"
chunk_size = 10000

output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
interval_end_years = [2016]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

tile_id = '00N_020E'

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/40000_pixels/{run_date}/"

adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"

# Spreadsheet for state_node meanings
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v030_20250430"

zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/zarr/{run_date}/"

adm0_zarr_name = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/20250604/global_GADM41_adm0_20250604.zarr"
pixel_area_zarr_name = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/20250604/global_pixel_area_20250604.zarr"

In [ ]:
%%time

print(f"Reading inputs that apply to all intervals: {timestr()}")

adm0_uris = list_folder_uris(adm0_folder)
pixel_area_uris = list_folder_uris(pixel_area_folder)

print("adm0_folder:", adm0_folder)
print(adm0_uris[0])
print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")
print("pixel_area_folder:", pixel_area_folder)
print(pixel_area_uris[0])
print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# Shouldn't need to run again! Did this once already!
print(f"   Reading adm0: {timestr()}")
adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
print(f"   Reading pixel_area: {timestr()}")
pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)

print("adm0_xarray_chunks:", adm0_xarray_chunks)
print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

print(f"   zarring adm0: {timestr()}")
adm0_xarray_chunks.to_zarr(adm0_zarr_name, mode='w')
print(f"   zarring pixel area: {timestr()}")
pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_name, mode='w')
print(f"Done creating zarrs for inputs that apply to all intervals: {timestr()}")

In [ ]:
pixel_area = xr.open_zarr(pixel_area_zarr_name).band_data
pixel_area

In [ ]:
adm0 = xr.open_zarr(adm0_zarr_name).band_data
adm0

In [ ]:
pixel_area_aligned, adm0_aligned = xr.align(pixel_area, adm0, join="inner")

In [ ]:
print(pixel_area_aligned.coords['y'])
print(adm0_aligned.coords['y'])

In [ ]:
%%time

area_sum = xarray_reduce(
    pixel_area_aligned,
    adm0_aligned,
    func='sum',
    expected_groups=(gadm_adm0_ids),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
).compute()

In [ ]:
sparse_data = area_sum.data

dim_names = area_sum.dims
indices = sparse_data.coords  # tuple of arrays with indices into each dim
values = sparse_data.data     # non-zero values

# Step 4: Map dimension indices to coordinate values
coord_dict = {
    dim: area_sum.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
coord_dict["value"] = values

df = pd.DataFrame(coord_dict)
df